# Clean SimCLR Pipeline for `light_curves.csv` (Tabular)
This notebook is a **cleaned + de-duplicated** version of your previous file. It contains:

1. **Data loading + preprocessing** (duplicates, missing values, categorical handling, scaling)  
2. **SimCLR training** (PyTorch, NT-Xent loss, early stopping, collapse checks)  
3. **Embedding extraction**  
4. **Downstream classifier training** (if labels are available)  
5. **Evaluation metrics**: Accuracy, Precision/Recall/F1 (macro/weighted), Confusion Matrix, Classification Report (and ROC-AUC for binary)

> You can run SimCLR without labels. The classifier + metrics section needs labels.


In [28]:
# =========================
# 0) Config
# =========================
import os
from dataclasses import dataclass

@dataclass
class CFG:
    # Data
    CSV_PATH: str = "./outputs/selected_features_full.csv"          # <-- change if needed
    ID_COL: str = "oid"                         # object id column (optional but recommended)
    LABEL_COL: str | None = None                # e.g. "transient_type" (set to None if unlabeled)

    # Columns
    # If FEATURE_COLS is None, we auto-select numeric columns except ID/LABEL.
    FEATURE_COLS: list[str] | None = None
    CATEGORICAL_COLS: list[str] = ("fid",)      # columns treated as categorical
    BINARY_COLS: list[str] = ("isdiffpos",)     # columns treated as binary (-1/1 or 0/1)

    # SimCLR training
    batch_size: int = 512
    epochs: int = 200
    lr: float = 1e-3
    weight_decay: float = 1e-6
    temperature: float = 0.2

    # Augmentations (continuous only)
    noise_std: float = 0.05
    drop_prob: float = 0.10
    scale_jitter: float = 0.05

    # Early stopping
    patience: int = 15
    min_delta: float = 1e-4

    # Model
    hidden: int = 256
    embed_dim: int = 128
    proj_dim: int = 128
    dropout: float = 0.1

    # Outputs
    OUT_DIR: str = "outputs_simclr_clean"
    MODEL_PATH: str = "simclr_encoder.pt"
    SCALER_PATH: str = "scaler.pkl"
    EMB_PATH: str = "embeddings.npy"

cfg = CFG()

os.makedirs(cfg.OUT_DIR, exist_ok=True)
print("Output dir:", cfg.OUT_DIR)


Output dir: outputs_simclr_clean


In [20]:
# =========================
# 1) Imports
# =========================
import numpy as np
import pandas as pd
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cpu


In [31]:
# =========================
# 2) Load preprocessed data
# =========================
df = pd.read_csv(cfg.CSV_PATH)
print("Loaded shape:", df.shape)
print("Columns:", list(df.columns))

# Exclude ID and label columns
excluded = set(c for c in [cfg.ID_COL, cfg.LABEL_COL] if c is not None)

# Feature columns = everything except excluded
FEATURE_COLS = [c for c in df.columns if c not in excluded]

print("Using preprocessed features (n=%d):" % len(FEATURE_COLS))
print(FEATURE_COLS)

# Build feature matrix (already clean & numeric)
X = df[FEATURE_COLS].to_numpy(dtype=np.float32)
print("Feature matrix X shape:", X.shape)

# Optional labels (for downstream classifier only)
# Try to auto-detect a separate labeled file 'labeled_dataset.csv' if present
y = None
label_file = 'labeled_dataset.csv'
if os.path.exists(label_file):
    df_labels = pd.read_csv(label_file)
    if 'transient_type' in df_labels.columns:
        # Attempt to align by common identifying columns if available
        possible_keys = ['mjd','fid','magpsf','sigmapsf','ra','dec']
        common_keys = [c for c in possible_keys if c in df.columns and c in df_labels.columns]
        if len(common_keys) >= 2:
            merged = df.merge(df_labels[common_keys + ['transient_type']], on=common_keys, how='left', suffixes=(None, '_lbl'))
            if 'transient_type' in merged.columns and merged['transient_type'].notna().sum() > 0:
                y = merged['transient_type'].astype(str).to_numpy()
                cfg.LABEL_COL = 'transient_type'
                print('Loaded labels from labeled_dataset.csv via merge on:', common_keys)
            else:
                # fallback: if lengths match, assume same ordering
                if len(df_labels) == len(df) and 'transient_type' in df_labels.columns:
                    y = df_labels['transient_type'].astype(str).to_numpy()
                    cfg.LABEL_COL = 'transient_type'
                    print('Loaded labels from labeled_dataset.csv by index (same length).')
                else:
                    print('Found labeled_dataset.csv but could not align labels automatically.')
        else:
            # Not enough common keys -> try index-based fallback
            if len(df_labels) == len(df) and 'transient_type' in df_labels.columns:
                y = df_labels['transient_type'].astype(str).to_numpy()
                cfg.LABEL_COL = 'transient_type'
                print('Loaded labels from labeled_dataset.csv by index (same length).')
            else:
                print('Found labeled_dataset.csv but it lacks a usable column or alignment keys.')
    else:
        print('labeled_dataset.csv present but missing "transient_type" column.')
else:
    # If user explicitly set LABEL_COL in cfg, use that column from df (original behavior)
    if cfg.LABEL_COL is not None:
        if cfg.LABEL_COL not in df.columns:
            raise ValueError(f"LABEL_COL '{cfg.LABEL_COL}' not found in CSV.")
        y = df[cfg.LABEL_COL].to_numpy()
        print('Loaded labels from main CSV column:', cfg.LABEL_COL)
    else:
        print('No labeled_dataset.csv found and cfg.LABEL_COL is None; continuing without labels.')

if y is not None:
    print('Label distribution:')
    print(pd.Series(y).value_counts())

# Train/validation split for SimCLR (labels NOT used here)
idx = np.arange(len(X))
idx_train, idx_val = train_test_split(
    idx, test_size=0.2, random_state=42, shuffle=True
)

X_train = X[idx_train]
X_val   = X[idx_val]

print("Train samples:", len(X_train))
print("Validation samples:", len(X_val))


Loaded shape: (20000, 7)
Columns: ['mjd', 'fid', 'magpsf', 'sigmapsf', 'ra', 'dec', 'isdiffpos']
Using preprocessed features (n=7):
['mjd', 'fid', 'magpsf', 'sigmapsf', 'ra', 'dec', 'isdiffpos']
Feature matrix X shape: (20000, 7)
Loaded labels from labeled_dataset.csv via merge on: ['mjd', 'fid', 'magpsf', 'sigmapsf', 'ra', 'dec']
Label distribution:
nan                     19949
Supernova_Ia               18
AGN_Flare                  14
Cataclysmic_Variable       10
Kilonova                    6
TDE                         3
Name: count, dtype: int64
Train samples: 16000
Validation samples: 4000


In [32]:
# =========================
# 3) SimCLR dataset + augmentations (continuous only)
# =========================
class SimCLRTabularDataset(Dataset):
    def __init__(self, X: np.ndarray, noise_std: float, drop_prob: float, scale_jitter: float):
        self.X = X.astype(np.float32)
        self.noise_std = noise_std
        self.drop_prob = drop_prob
        self.scale_jitter = scale_jitter

    def _augment(self, x: np.ndarray) -> np.ndarray:
        x = x.copy()

        # Gaussian noise
        x += np.random.normal(0.0, self.noise_std, size=x.shape).astype(np.float32)

        # Feature dropout (mask to 0)
        drop_mask = (np.random.rand(*x.shape) > self.drop_prob).astype(np.float32)
        x *= drop_mask

        # Multiplicative jitter
        x *= (1.0 + np.random.normal(0.0, self.scale_jitter, size=x.shape).astype(np.float32))

        return x

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        v1 = self._augment(x)
        v2 = self._augment(x)
        return torch.from_numpy(v1), torch.from_numpy(v2)

train_ds = SimCLRTabularDataset(X_train, cfg.noise_std, cfg.drop_prob, cfg.scale_jitter)
val_ds   = SimCLRTabularDataset(X_val,   cfg.noise_std, cfg.drop_prob, cfg.scale_jitter)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, drop_last=True)

print("Batches (train/val):", len(train_loader), len(val_loader))


Batches (train/val): 31 7


In [33]:
# =========================
# 4) Encoder + projection head
# =========================
class MLPEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden: int, embed_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, embed_dim),
        )

    def forward(self, x):
        return self.net(x)

class ProjectionHead(nn.Module):
    def __init__(self, embed_dim: int, proj_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, proj_dim),
        )

    def forward(self, z):
        return self.net(z)

class SimCLR(nn.Module):
    def __init__(self, in_dim: int, hidden: int, embed_dim: int, proj_dim: int, dropout: float):
        super().__init__()
        self.encoder = MLPEncoder(in_dim, hidden, embed_dim, dropout)
        self.proj = ProjectionHead(embed_dim, proj_dim)

    def forward(self, x):
        h = self.encoder(x)
        z = self.proj(h)
        z = F.normalize(z, dim=1)
        return h, z

in_dim = X_train.shape[1]
model = SimCLR(in_dim, cfg.hidden, cfg.embed_dim, cfg.proj_dim, cfg.dropout).to(DEVICE)
print("Model params:", sum(p.numel() for p in model.parameters()))


Model params: 134784


In [24]:
# =========================
# 5) NT-Xent loss
# =========================
def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float) -> torch.Tensor:
    """NT-Xent for a batch. z1,z2 are L2-normalized."""
    batch_size = z1.size(0)
    z = torch.cat([z1, z2], dim=0)  # (2B, D)

    sim = torch.mm(z, z.t()) / temperature  # (2B, 2B)
    # mask out self similarity
    mask = torch.eye(2 * batch_size, device=sim.device).bool()
    sim.masked_fill_(mask, -1e9)

    # positives: i-th sample in z1 matches i-th in z2
    positives = torch.cat([torch.diag(sim, batch_size), torch.diag(sim, -batch_size)], dim=0)  # (2B,)

    # denominator: logsumexp over all except self
    loss = -positives + torch.logsumexp(sim, dim=1)
    return loss.mean()

# quick sanity check
x1, x2 = next(iter(train_loader))
with torch.no_grad():
    _, z1 = model(x1.to(DEVICE))
    _, z2 = model(x2.to(DEVICE))
print("Sanity loss:", float(nt_xent_loss(z1, z2, cfg.temperature)))


Sanity loss: 7.200313091278076


In [34]:
import os, pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# =========================
# 0) Config
# =========================
class CFG:
    CSV_PATH = "light_curves.csv"
    OUT_DIR  = "outputs_simclr_clean"
    ID_COL   = "oid"          # set None if not available
    LABEL_COL = None          # optional (for later classifier)
    
    # columns in your raw CSV
    CAT_COL = "fid"
    BIN_COL = "isdiffpos"
    CONT_COLS = ["mjd", "magpsf", "sigmapsf", "ra", "dec"]
    
    # simclr training
    batch_size = 512
    epochs = 200
    lr = 3e-4
    weight_decay = 1e-4
    temperature = 0.2
    
    # early stopping
    patience = 15
    min_delta = 1e-4
    
    # augmentation strengths (continuous only)
    noise_std = 0.05
    drop_prob = 0.10
    scale_jitter = 0.05
    
    # model sizes
    hidden = 256
    emb_dim = 128
    proj_dim = 128

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


# =========================
# 1) Preprocess (proper for SimCLR)
# =========================
df = pd.read_csv(cfg.CSV_PATH)
print("Raw shape:", df.shape)
print("Columns:", list(df.columns))

# --- basic cleanup ---
df = df.drop_duplicates().reset_index(drop=True)

# --- validate continuous columns exist ---
for c in cfg.CONT_COLS:
    if c not in df.columns:
        raise ValueError(f"Missing continuous column: {c}")

# --- handle isdiffpos to 0/1 ---
if cfg.BIN_COL in df.columns:
    vals = set(pd.unique(df[cfg.BIN_COL].dropna()))
    if vals.issubset({-1, 1}):
        df[cfg.BIN_COL] = (df[cfg.BIN_COL] == 1).astype(int)
    else:
        # fallback: >0 -> 1 else 0
        df[cfg.BIN_COL] = (df[cfg.BIN_COL].astype(float) > 0).astype(int)
else:
    # if missing, create a dummy column of zeros (optional)
    df[cfg.BIN_COL] = 0

# --- fill NaNs lightly (safe) ---
# median for continuous
for c in cfg.CONT_COLS:
    df[c] = df[c].fillna(df[c].median())

# mode for fid if missing
if cfg.CAT_COL in df.columns:
    if df[cfg.CAT_COL].isna().any():
        mode_val = df[cfg.CAT_COL].mode()
        df[cfg.CAT_COL] = df[cfg.CAT_COL].fillna(mode_val.iloc[0] if len(mode_val) else 0)
else:
    raise ValueError(f"Missing categorical column: {cfg.CAT_COL}")

# --- one-hot encode fid ---
df_proc = pd.get_dummies(df, columns=[cfg.CAT_COL], drop_first=False)

# --- standardize continuous ONLY ---
scaler = StandardScaler()
df_proc[cfg.CONT_COLS] = scaler.fit_transform(df_proc[cfg.CONT_COLS])

# --- build final feature columns (continuous + fid one-hot + binary) ---
excluded = set()
if cfg.ID_COL is not None and cfg.ID_COL in df_proc.columns:
    excluded.add(cfg.ID_COL)
if cfg.LABEL_COL is not None and cfg.LABEL_COL in df_proc.columns:
    excluded.add(cfg.LABEL_COL)

feature_cols = [c for c in df_proc.columns if c not in excluded]

# Identify which columns are continuous vs discrete for augmentation masking
cont_cols = cfg.CONT_COLS[:]  # already scaled
# one-hot columns from fid look like "fid_1", "fid_2", ...
onehot_cols = [c for c in feature_cols if c.startswith(cfg.CAT_COL + "_")]
bin_cols = [cfg.BIN_COL] if cfg.BIN_COL in feature_cols else []

# final ordering: cont + onehot + bin (nice and consistent)
feature_cols = cont_cols + onehot_cols + bin_cols

X = df_proc[feature_cols].to_numpy(dtype=np.float32)
print("Processed X shape:", X.shape)
print("Feature count:", len(feature_cols))
print("Continuous:", cont_cols)
print("One-hot:", onehot_cols[:10], "..." if len(onehot_cols) > 10 else "")
print("Binary:", bin_cols)

# save preprocessing artifacts
with open(os.path.join(cfg.OUT_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
with open(os.path.join(cfg.OUT_DIR, "feature_cols.pkl"), "wb") as f:
    pickle.dump(feature_cols, f)
print("Saved scaler + feature cols to:", cfg.OUT_DIR)


# =========================
# 2) SimCLR Dataset (augment continuous only)
# =========================
class SimCLRTabularDataset(Dataset):
    def __init__(self, X: np.ndarray, n_cont: int, noise_std=0.05, drop_prob=0.1, scale_jitter=0.05):
        self.X = X
        self.n_cont = n_cont
        self.noise_std = noise_std
        self.drop_prob = drop_prob
        self.scale_jitter = scale_jitter

    def _augment(self, x: torch.Tensor) -> torch.Tensor:
        """
        Augment ONLY first n_cont continuous features.
        Keep one-hot + binary unchanged (very important).
        """
        x2 = x.clone()
        cont = x2[:self.n_cont]

        # gaussian noise
        cont = cont + torch.randn_like(cont) * self.noise_std

        # feature dropout (mask some continuous features)
        mask = (torch.rand_like(cont) > self.drop_prob).float()
        cont = cont * mask

        # scale jitter
        cont = cont * (1.0 + torch.randn_like(cont) * self.scale_jitter)

        x2[:self.n_cont] = cont
        return x2

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])
        v1 = self._augment(x)
        v2 = self._augment(x)
        return v1, v2

# train/val split (labels not needed)
idx = np.arange(len(X))
idx_train, idx_val = train_test_split(idx, test_size=0.2, random_state=42, shuffle=True)

ds_train = SimCLRTabularDataset(
    X[idx_train], n_cont=len(cont_cols),
    noise_std=cfg.noise_std, drop_prob=cfg.drop_prob, scale_jitter=cfg.scale_jitter
)
ds_val = SimCLRTabularDataset(
    X[idx_val], n_cont=len(cont_cols),
    noise_std=cfg.noise_std, drop_prob=cfg.drop_prob, scale_jitter=cfg.scale_jitter
)

train_loader = DataLoader(ds_train, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
val_loader   = DataLoader(ds_val, batch_size=cfg.batch_size, shuffle=False, drop_last=False)

print("Train/Val batches:", len(train_loader), len(val_loader))


# =========================
# 3) SimCLR Model (MLP encoder + projection head)
# =========================
class MLPEncoder(nn.Module):
    def __init__(self, in_dim, hidden=256, emb_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, emb_dim)
        )

    def forward(self, x):
        return self.net(x)

class ProjectionHead(nn.Module):
    def __init__(self, emb_dim=128, proj_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, proj_dim)
        )

    def forward(self, h):
        return self.net(h)

class SimCLR(nn.Module):
    def __init__(self, in_dim, hidden=256, emb_dim=128, proj_dim=128):
        super().__init__()
        self.encoder = MLPEncoder(in_dim, hidden, emb_dim)
        self.projector = ProjectionHead(emb_dim, proj_dim)

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, z

def nt_xent_loss(z1, z2, temperature=0.2):
    """
    Standard SimCLR NT-Xent loss (cosine similarity).
    """
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    N = z1.size(0)

    z = torch.cat([z1, z2], dim=0)  # (2N, d)
    sim = torch.mm(z, z.t()) / temperature  # (2N, 2N)

    # mask self-similarity
    mask = torch.eye(2*N, device=z.device).bool()
    sim.masked_fill_(mask, -1e9)

    # positives: i <-> i+N
    pos = torch.cat([torch.diag(sim, N), torch.diag(sim, -N)], dim=0)

    # denominator: logsumexp over row
    denom = torch.logsumexp(sim, dim=1)
    loss = -pos + denom
    return loss.mean()

model = SimCLR(
    in_dim=X.shape[1],
    hidden=cfg.hidden,
    emb_dim=cfg.emb_dim,
    proj_dim=cfg.proj_dim
).to(DEVICE)

print("Model params:", sum(p.numel() for p in model.parameters()))


# =========================
# 4) Train loop (stable + early stopping + collapse check on h)
# =========================
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)

def embedding_collapse_std(model: SimCLR, loader: DataLoader, n_batches: int = 5) -> float:
    """
    Check collapse using encoder output h (not projector z).
    If std ~ 0, embeddings collapsed.
    """
    model.eval()
    hs = []
    with torch.no_grad():
        for i, (a, b) in enumerate(loader):
            if i >= n_batches:
                break
            h, _ = model(a.to(DEVICE))
            hs.append(h.detach().cpu())
    h_all = torch.cat(hs, dim=0)
    return float(h_all.std(dim=0).mean())

# sanity loss
model.eval()
with torch.no_grad():
    v1, v2 = next(iter(train_loader))
    _, z1 = model(v1.to(DEVICE))
    _, z2 = model(v2.to(DEVICE))
    print("Sanity loss:", float(nt_xent_loss(z1, z2, cfg.temperature).cpu()))

best_val = float("inf")
epochs_no_improve = 0
best_path = os.path.join(cfg.OUT_DIR, "simclr_encoder.pt")

for epoch in range(1, cfg.epochs + 1):
    # ---- train ----
    model.train()
    tr_losses = []

    for v1, v2 in tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.epochs}", leave=False):
        v1, v2 = v1.to(DEVICE), v2.to(DEVICE)

        _, z1 = model(v1)
        _, z2 = model(v2)
        loss = nt_xent_loss(z1, z2, cfg.temperature)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # stability
        opt.step()

        tr_losses.append(loss.item())

    sched.step()
    tr_loss = float(np.mean(tr_losses))

    # ---- val ----
    model.eval()
    val_losses = []
    with torch.no_grad():
        for v1, v2 in val_loader:
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            _, z1 = model(v1)
            _, z2 = model(v2)
            val_losses.append(nt_xent_loss(z1, z2, cfg.temperature).item())
    val_loss = float(np.mean(val_losses))

    e_std = embedding_collapse_std(model, val_loader)

    print(f"Epoch {epoch:03d} | train {tr_loss:.4f} | val {val_loss:.4f} | embed_std(h) {e_std:.4f}")

    # ---- early stopping ----
    if val_loss < best_val - cfg.min_delta:
        best_val = val_loss
        epochs_no_improve = 0
        torch.save(model.encoder.state_dict(), best_path)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= cfg.patience:
            print(f"Early stopping: no val improvement for {cfg.patience} epochs.")
            break

print("Saved best encoder to:", best_path)


Device: cpu
Raw shape: (20000, 10)
Columns: ['oid', 'mjd', 'fid', 'mag', 'e_mag', 'magpsf', 'sigmapsf', 'ra', 'dec', 'isdiffpos']
Processed X shape: (19736, 8)
Feature count: 8
Continuous: ['mjd', 'magpsf', 'sigmapsf', 'ra', 'dec']
One-hot: ['fid_1', 'fid_2'] 
Binary: ['isdiffpos']
Saved scaler + feature cols to: outputs_simclr_clean
Train/Val batches: 30 8
Model params: 134016
Sanity loss: 6.77300500869751


Epoch 1/200:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 001 | train 5.2854 | val 4.4829 | embed_std(h) 0.1534


Epoch 2/200:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 002 | train 4.3806 | val 4.2084 | embed_std(h) 0.1686


...

Epoch 140 | train 3.3348 | val 3.3271 | embed_std(h) 0.2130


Epoch 141/200:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 141 | train 3.3353 | val 3.2852 | embed_std(h) 0.2151


Epoch 142/200:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 142 | train 3.3395 | val 3.3008 | embed_std(h) 0.2142
Early stopping: no val improvement for 15 epochs.
Saved best encoder to: outputs_simclr_clean\simclr_encoder.pt


In [26]:
# =========================
# 7) Extract embeddings for ALL samples
# =========================

# Rebuild encoder EXACTLY as during training
encoder = MLPEncoder(
    in_dim=X.shape[1],          # input feature dimension
    hidden=cfg.hidden,
    emb_dim=cfg.emb_dim
).to(DEVICE)

# Load trained encoder weights
encoder.load_state_dict(
    torch.load(os.path.join(cfg.OUT_DIR, "simclr_encoder.pt"), map_location=DEVICE)
)
encoder.eval()

# NOTE:
# X is already preprocessed + scaled in the pipeline
X_all = X.astype(np.float32)

# Batch-wise embedding extraction
embeddings = []
bs = 2048

with torch.no_grad():
    for i in range(0, len(X_all), bs):
        xb = torch.from_numpy(X_all[i:i + bs]).to(DEVICE)
        hb = encoder(xb).cpu().numpy()
        embeddings.append(hb)

embeddings = np.vstack(embeddings)

# Save embeddings
emb_path = os.path.join(cfg.OUT_DIR, "embeddings.npy")
np.save(emb_path, embeddings)

print("Embeddings shape:", embeddings.shape)
print("Saved embeddings to:", emb_path)


Embeddings shape: (19736, 128)
Saved embeddings to: outputs_simclr_clean\embeddings.npy


## Downstream classifier + accuracy metrics (requires labels)
If you have labels (e.g., `transient_type`), set `CFG.LABEL_COL` at the top and re-run.
This section trains a simple baseline classifier on embeddings and reports standard metrics.


In [27]:
# =========================
# 8) Classifier on embeddings (optional)
# =========================
if y is None:
    print("No labels provided. Set cfg.LABEL_COL to enable classifier training + metrics.")
else:
    # train/test split with labels
    X_tr, X_te, y_tr, y_te = train_test_split(emb, y, test_size=0.2, random_state=42, stratify=y)

    clf = LogisticRegression(max_iter=2000, n_jobs=None)
    clf.fit(X_tr, y_tr)

    y_pred = clf.predict(X_te)

    # --- metrics ---
    acc = accuracy_score(y_te, y_pred)
    prec_macro = precision_score(y_te, y_pred, average="macro", zero_division=0)
    rec_macro  = recall_score(y_te, y_pred, average="macro", zero_division=0)
    f1_macro   = f1_score(y_te, y_pred, average="macro", zero_division=0)

    prec_w = precision_score(y_te, y_pred, average="weighted", zero_division=0)
    rec_w  = recall_score(y_te, y_pred, average="weighted", zero_division=0)
    f1_w   = f1_score(y_te, y_pred, average="weighted", zero_division=0)

    print("Accuracy:", acc)
    print("Macro   Precision/Recall/F1:", prec_macro, rec_macro, f1_macro)
    print("Weighted Precision/Recall/F1:", prec_w, rec_w, f1_w)

    # Confusion matrix
    labels_sorted = np.unique(y_te)
    cm = confusion_matrix(y_te, y_pred, labels=labels_sorted)
    print("\nConfusion Matrix (rows=true, cols=pred):")
    print(pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted))

    print("\nClassification Report:")
    print(classification_report(y_te, y_pred, zero_division=0))

    # ROC-AUC for binary only
    if len(labels_sorted) == 2:
        proba = clf.predict_proba(X_te)[:, 1]
        # Need to binarize y for roc_auc_score
        y_bin = (y_te == labels_sorted[1]).astype(int)
        auc = roc_auc_score(y_bin, proba)
        print("ROC-AUC:", auc)

    # Save classifier
    with open(os.path.join(cfg.OUT_DIR, "classifier_logreg.pkl"), "wb") as f:
        pickle.dump({"clf": clf, "classes": clf.classes_}, f)
    print("Saved classifier to:", os.path.join(cfg.OUT_DIR, "classifier_logreg.pkl"))


No labels provided. Set cfg.LABEL_COL to enable classifier training + metrics.
